## 6. Combined fine-tuned RoBERTa w/ audio features

#### isolated to only include highest performing model of the 8 tested variants that included lyric-processing

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from transformers import RobertaTokenizer, RobertaModel
from tqdm.notebook import tqdm
from google.colab import files
import io
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')



df = pd.read_csv('/content/drive/MyDrive/w266_final_project/source/lyrics_df_cleaned.csv')


# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

df.head()

Mounted at /content/drive
Using device: cuda
GPU: NVIDIA L4


,Unnamed: 0,track_name,track_artist,valence,lyrics_snippet,track_popularity,track_album_id,track_album_name,track_album_release_date,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,tempo,duration_ms
0,0,Dance Monkey,Tones and I,0.513,"They say, ""Oh my god, I see the way you shine ...",100,0UywfDKYlyiu1b38DRrzYD,Dance Monkey (Stripped Back) / Dance Monkey,2019-10-17,0.824,0.588,6,-6.400,0,0.0924,0.69200,0.000104,0.1490,98.027,209438
1,32,ROXANNE,Arizona Zervas,0.457,"All for the 'Gram Bitches love the 'Gram Oh, w...",99,6HJDrXs0hpebaRFKA1sF90,ROXANNE,2019-10-10,0.621,0.601,6,-5.616,0,0.1480,0.05220,0.000000,0.4600,116.735,163636
2,1056,The Box,Roddy Ricch,0.642,Pullin' out the coupe at the lot Told 'em fuck...,98,52u4anZbHd6UInnmHRFzba,Please Excuse Me For Being Antisocial,2019-12-06,0.896,0.586,10,-6.687,0,0.0559,0.10400,0.000000,0.7900,116.971,196653
3,33824,Blinding Lights,The Weeknd,0.345,Yeah I've been tryna call I've been on my own...,98,2ZfHkwHuoAZrlz7RMj0PDz,Blinding Lights,2019-11-29,0.513,0.796,1,-4.075,1,0.0629,0.00147,0.000209,0.0938,171.017,201573
4,66592,Memories,Maroon 5,0.575,Here's to the ones that we got Cheers to the w...,98,3nR9B40hYLKLcR0Eph3Goc,Memories,2019-09-20,0.764,0.320,11,-7.209,1,0.0546,0.83700,0.000000,0.0822,91.019,189486


**step 2**: define target, features, and lyrics

In [2]:

y = df['valence']

audio_features = [
    'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'tempo'
]
X_audio = df[audio_features]

lyrics_series = df['lyrics_snippet'].fillna('')

print(f"Audio features shape: {X_audio.shape}")
print(f"Target shape: {y.shape}")
print(f"Lyrics count: {len(lyrics_series)}")

Audio features shape: (3717, 10)
Target shape: (3717,)
Lyrics count: 3717


**step 3**: tokenize lyrics

In [3]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

X_lyrics_tokenized = tokenizer(
    lyrics_series.tolist(),
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors='pt'
)

print(f"input_ids shape: {X_lyrics_tokenized['input_ids'].shape}")
print(f"attention_mask shape: {X_lyrics_tokenized['attention_mask'].shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

input_ids shape: torch.Size([3717, 512])
attention_mask shape: torch.Size([3717, 512])


**step 4:** split inputs & labels

In [4]:

y_tensor = torch.tensor(y.values, dtype=torch.float32)

# split 1: 70% train, 30% temp
X_audio_train, X_audio_temp, \
X_input_ids_train, X_input_ids_temp, \
X_attn_mask_train, X_attn_mask_temp, \
y_train, y_temp = train_test_split(
    X_audio,
    X_lyrics_tokenized['input_ids'],
    X_lyrics_tokenized['attention_mask'],
    y_tensor,
    test_size=0.3,
    random_state=42
)

#split 2: 30% temp --> 15% test, val
X_audio_val, X_audio_test, \
X_input_ids_val, X_input_ids_test, \
X_attn_mask_val, X_attn_mask_test, \
y_val, y_test = train_test_split(
    X_audio_temp,
    X_input_ids_temp,
    X_attn_mask_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)


**step 5:** scale audio features to normalize variance using standardscaler


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_audio_train_scaled = scaler.fit_transform(X_audio_train)
X_audio_val_scaled = scaler.transform(X_audio_val)
X_audio_test_scaled = scaler.transform(X_audio_test)

print(f"post-scaling feature shape: {X_audio_train_scaled.shape}")

post-scaling feature shape: (2601, 10)


**step 6:** create bespoke class to store combined inputs/outputs. also, create necessary dataloaders for training,testing, and validation values

In [6]:
class CombinedDataset(Dataset):
    def __init__(self, input_ids, attention_mask, audio_features, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.audio_features = torch.tensor(audio_features, dtype=torch.float32)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'audio_features': self.audio_features[idx],
            'labels': self.labels[idx]
        }

batch_size = 16
train_dataset = CombinedDataset(X_input_ids_train, X_attn_mask_train, X_audio_train_scaled, y_train)
val_dataset = CombinedDataset(X_input_ids_val, X_attn_mask_val, X_audio_val_scaled, y_val)
test_dataset = CombinedDataset(X_input_ids_test, X_attn_mask_test, X_audio_test_scaled, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

**step 7:** create model class

In [7]:
import torch.nn as nn
class CombinedFineTunedRobertaModel(nn.Module):
    def __init__(self, num_audio_features=10):
        super().__init__()

        # initiate roBERTa - but keep all layers trainable & remove pooling classification layer to enable fine tuning
        self.roberta = RobertaModel.from_pretrained('roberta-base', add_pooling_layer=False)

        # combined feature size: 768 (from lyrics) + 10 (audio) = 778
        combined_size = self.roberta.config.hidden_size + num_audio_features

        # implement regression head w/ following parameters
        self.regressor = nn.Sequential(
            nn.Linear(combined_size, combined_size),
            nn.ReLU(), ## non-linear pattern learning
            nn.Dropout(0.1),
            nn.Linear(combined_size, 1)
        )

    def forward(self, input_ids, attention_mask, audio_features):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        combined = torch.cat((cls_output, audio_features), dim=1)  # [batch, 778]

        return self.regressor(combined).squeeze(1)

# Initialize model
model = CombinedFineTunedRobertaModel(num_audio_features=10)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Total parameters: 124,661,881
Trainable parameters: 124,661,881


**step 8:** train model and evaluate results

In [8]:
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.MSELoss()
num_epochs = 3

best_val_r2 = -float('inf')
best_epoch = 0

In [10]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        audio_features = batch['audio_features'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        predictions = model(input_ids, attention_mask, audio_features)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    val_preds, val_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            audio_features = batch['audio_features'].to(device)
            labels = batch['labels'].to(device)

            predictions = model(input_ids, attention_mask, audio_features)
            val_preds.extend(predictions.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_r2 = r2_score(val_labels, val_preds)
    val_rmse = np.sqrt(mean_squared_error(val_labels, val_preds))
    val_mae = mean_absolute_error(val_labels, val_preds)

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val R2: {val_r2:.4f} | RMSE: {val_rmse:.4f} | MAE: {val_mae:.4f}")

    # Save best model
    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_epoch = epoch + 1
        torch.save(model.state_dict(), 'best_combined_finetuned_roberta.pt')

Epoch 1/3:   0%|          | 0/163 [00:00<?, ?it/s]


Epoch 1/3
  Train Loss: 0.0458
  Val R2: 0.1373 | RMSE: 0.2078 | MAE: 0.1719


Epoch 2/3:   0%|          | 0/163 [00:00<?, ?it/s]


Epoch 2/3
  Train Loss: 0.0390
  Val R2: -0.0134 | RMSE: 0.2252 | MAE: 0.1831


Epoch 3/3:   0%|          | 0/163 [00:00<?, ?it/s]


Epoch 3/3
  Train Loss: 0.0317
  Val R2: 0.1746 | RMSE: 0.2033 | MAE: 0.1652


In [11]:
model.load_state_dict(torch.load('best_combined_finetuned_roberta.pt'))
model.eval()

test_preds, test_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        audio_features = batch['audio_features'].to(device)
        labels = batch['labels'].to(device)

        predictions = model(input_ids, attention_mask, audio_features)
        test_preds.extend(predictions.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

test_r2 = r2_score(test_labels, test_preds)
test_mse = mean_squared_error(test_labels, test_preds)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(test_labels, test_preds)

print("=" * 60)
print("COMBINED FINE-TUNED RoBERTa + AUDIO - TEST RESULTS")
print("=" * 60)
print(f"  R2:   {test_r2:.4f}")
print(f"  MSE:  {test_mse:.4f}")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  MAE:  {test_mae:.4f}")
print("=" * 60)

COMBINED FINE-TUNED RoBERTa + AUDIO - TEST RESULTS
  R2:   0.2334
  MSE:  0.0340
  RMSE: 0.1843
  MAE:  0.1458
